In [ ]:
# KALMAN NET CONTRACT FORENSICS v2.1 — ONE CELL / READ ONLY
# Diagnose why reconstructed_fixed4_net_return cannot be derived as raw +/- cost_proxy.
from google.colab import drive
drive.mount("/content/drive",force_remount=False)
from pathlib import Path
import numpy as np, pandas as pd, json

ROOT=Path("/content/drive/MyDrive/US_ETF/model_lab_v1/results")
AUD=ROOT/"open_revalidation_v1/open_revalidation_trade_audit.parquet"
LED=ROOT/"exit_policy_v1_0_pre2026/exit_policy_v1_0_1_trade_ledger.parquet"
BAR=ROOT/"exit_policy_v1_0_pre2026/exit_policy_v1_0_1_bar_returns.parquet"
for p in [AUD,LED,BAR]: print("[FILE]",p,p.exists())
a=pd.read_parquet(AUD); l=pd.read_parquet(LED); b=pd.read_parquet(BAR)
print("\n[SHAPES]",{"audit":a.shape,"ledger":l.shape,"bar_returns":b.shape})
print("\n[AUDIT COLS]\n",a.columns.tolist())
print("\n[LEDGER COLS]\n",l.columns.tolist())
print("\n[BAR COLS]\n",b.columns.tolist())

# Existing 20-row legacy subset
k=a["reconstructed_fixed4_net_return"].notna()
cols=[c for c in ["policy","fold","symbol","entry_timestamp","exit_timestamp","entry_price_iex","fixed4_exit_price_iex","reconstructed_fixed4_raw_return","cost_proxy","reconstructed_fixed4_net_return","net_return","gross_return","weight","side"] if c in a]
z=a.loc[k,cols].copy()
print("\n[LEGACY 20]\n",z.to_string(index=False))

# algebraic diagnostics
for c in ["reconstructed_fixed4_raw_return","cost_proxy","reconstructed_fixed4_net_return","net_return"]:
 if c in a: a[c]=pd.to_numeric(a[c],errors="coerce")
if all(c in a for c in ["reconstructed_fixed4_raw_return","reconstructed_fixed4_net_return","cost_proxy"]):
 q=a.loc[k,["reconstructed_fixed4_raw_return","reconstructed_fixed4_net_return","cost_proxy"]].copy()
 q["net_minus_raw"]=q["reconstructed_fixed4_net_return"]-q["reconstructed_fixed4_raw_return"]
 q["implied_cost_abs"]=q["reconstructed_fixed4_raw_return"]-q["reconstructed_fixed4_net_return"]
 q["ratio_implied_to_cost"]=q["implied_cost_abs"]/q["cost_proxy"].replace(0,np.nan)
 print("\n[NET ALGEBRA]\n",q.to_string(index=False))
 print("\n[NET ALGEBRA SUMMARY]\n",q.describe().to_string())

# Find likely corresponding fields/contracts in original ledger and bar-return tables.
terms=["return","ret","cost","fee","slip","weight","gross","net","entry","exit","holding","policy","symbol","timestamp","fold"]
for name,df in [("LEDGER",l),("BAR",b)]:
 hits=[c for c in df.columns if any(t in c.lower() for t in terms)]
 print(f"\n[{name} LIKELY CONTRACT COLS]\n",hits)
 print(df[hits[:30]].head(12).to_string(index=False) if hits else df.head(12).to_string(index=False))

# Attempt conservative key overlap inspection only; no writes.
common=[c for c in ["symbol","fold","entry_timestamp","exit_timestamp","entry_seq","exit_seq","policy"] if c in a.columns and c in l.columns]
print("\n[COMMON AUDIT↔LEDGER KEYS]",common)
for c in common:
 print(c,"audit_nunique=",a[c].nunique(dropna=False),"ledger_nunique=",l[c].nunique(dropna=False))

print("\nREAD ONLY: no canonical files modified.")
print("NEXT: use this output to recover the exact historical net-return/cost contract; do not rerun v2.0 yet.")
